# Get citing & cited opinions metadata for Bankruptcy Courts

In the previous experiments, we've primarily focused on SCOTUS. This is to expand the dataset to bankruptcy courts in the Federal jurisdiction, including the Bankruptcy appeals courts and the Bankruptcy panel courts. 

This notebook documents the steps I undertook to:
1. Identify the Federal bankruptcy courts to include
2. Use Django Shell to sample target cases in the target courts from CL Replica
3. Use Django Shell to get all cases that cited the sampled target cases - these are the citing cases
4. Use Django Shell to get all authorities for the citing cases - these are the cited cases, including ones in the specified courts (target cases) and the ones not in the specified courts
5. Use Django Shell to get related metadata for the citing and cited cases

# Import Libaries

In [1]:
import json

import numpy as np
import pandas as pd

# Load Court Hierarchy data

In [2]:
df = pd.read_csv("../experiments_624/court_hierarchy_flp.csv")
df.head()

,id,full_name,jurisdiction,jurisdiction_name,jurisdiction_type,jurisdiction_state,appeals_to_full_name,appeals_to_id,note
0,scotus,Supreme Court of the United States,F,Federal Appellate,Federal,Federal,NaN,NaN,NaN
1,cafc,Court of Appeals for the Federal Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN
2,ca1,Court of Appeals for the First Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN
3,ca2,Court of Appeals for the Second Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN
4,ca3,Court of Appeals for the Third Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN


In [3]:
df["jurisdiction_name"].value_counts()

jurisdiction_name
Federal Bankruptcy          141
Federal District             94
State Appellate              57
State Supreme                52
Federal Appellate            14
Federal Bankruptcy Panel      8
State Trial                   6
Territory Supreme             5
Territory Trial               4
Territory Appellate           2
Name: count, dtype: int64

In [4]:
fed_bankruptcy = df[df["jurisdiction_name"] == "Federal Bankruptcy"]
fed_bankruptcy["full_name"]

20          United States Bankruptcy Court, M.D. Alabama
21          United States Bankruptcy Court, N.D. Alabama
22          United States Bankruptcy Court, S.D. Alabama
23             United States Bankruptcy Court, D. Alaska
24             United States Bankruptcy Court, D. Alaska
                             ...                        
158    United States Bankruptcy Court, Northern Maria...
159      United States Bankruptcy Court, D. Rhode Island
160       United States Bankruptcy Court, D. Puerto Rico
161       United States Bankruptcy Court, D. Puerto Rico
162    United States Bankruptcy Court, D. Virgin Islands
Name: full_name, Length: 141, dtype: object

In [5]:
fed_bankruptcy_panel = df[df["jurisdiction_name"] == "Federal Bankruptcy Panel"]
fed_bankruptcy_panel["full_name"]

14      Bankruptcy Appellate Panel of the First Circuit
15     Bankruptcy Appellate Panel of the Second Circuit
16      Bankruptcy Appellate Panel of the Sixth Circuit
17    United States Bankruptcy Appellate Panel for t...
18    United States Bankruptcy Appellate Panel for t...
19      Bankruptcy Appellate Panel of the Tenth Circuit
72                 Bankruptcy Appellate Panel, D. Maine
76          Bankruptcy Appellate Panel of Massachusetts
Name: full_name, dtype: object

In [6]:
fed_bankruptcy["jurisdiction_state"].value_counts()

jurisdiction_state
California                  8
Oklahoma                    6
Tennessee                   6
Iowa                        4
New York                    4
Ohio                        4
Arkansas                    4
Missouri                    4
Michigan                    4
Texas                       4
Washington                  4
Kentucky                    4
Pennsylvania                3
Louisiana                   3
Alabama                     3
Illinois                    3
Georgia                     3
Florida                     3
North Carolina              3
Virginia                    2
Utah                        2
West Virginia               2
South Dakota                2
Wisconsin                   2
Rhode Island                2
Wyoming                     2
Guam                        2
Oregon                      2
Northern Mariana Islands    2
Puerto Rico                 2
North Dakota                2
Nebraska                    2
New Mexico           

In [7]:
court_ids = fed_bankruptcy["id"].to_list() + fed_bankruptcy_panel["id"].to_list()
len(court_ids)

149

In [8]:
court_ids

['almb',
 'alnb',
 'alsb',
 'akb',
 'akb',
 'arb',
 'arb',
 'areb',
 'areb',
 'arwb',
 'arwb',
 'cacb',
 'cacb',
 'caeb',
 'caeb',
 'canb',
 'canb',
 'casb',
 'casb',
 'cob',
 'cob',
 'ctb',
 'deb',
 'dcb',
 'flmb',
 'flnb',
 'flsb',
 'gamb',
 'ganb',
 'gasb',
 'hib',
 'hib',
 'idb',
 'idb',
 'ilcb',
 'ilnb',
 'ilsb',
 'innb',
 'insb',
 'ianb',
 'ianb',
 'iasb',
 'iasb',
 'ksb',
 'ksb',
 'kyeb',
 'kyeb',
 'kywb',
 'kywb',
 'laeb',
 'lamb',
 'lawb',
 'meb',
 'meb',
 'mdb',
 'mab',
 'mab',
 'mieb',
 'mieb',
 'miwb',
 'miwb',
 'mnb',
 'mnb',
 'msnb',
 'mssb',
 'moeb',
 'moeb',
 'mowb',
 'mowb',
 'mtb',
 'mtb',
 'nebraskab',
 'nebraskab',
 'nvb',
 'nvb',
 'nhb',
 'njb',
 'nmb',
 'nmb',
 'nyeb',
 'nynb',
 'nysb',
 'nywb',
 'nceb',
 'ncmb',
 'ncwb',
 'ndb',
 'ndb',
 'ohnb',
 'ohnb',
 'ohsb',
 'ohsb',
 'okeb',
 'okeb',
 'oknb',
 'oknb',
 'okwb',
 'okwb',
 'orb',
 'orb',
 'paeb',
 'pamb',
 'pawb',
 'nhb',
 'rib',
 'scb',
 'sdb',
 'sdb',
 'tneb',
 'tneb',
 'tnmb',
 'tnmb',
 'tnwb',
 'tnwb',
 't

# Use Django Shell to get the list of target case cluster ids and citing case cluster ids

In [9]:
with open("data/citing_ids.txt", "r") as f:
    citing_ids = [line.strip() for line in f]
len(citing_ids)

1493

In [10]:
with open("data/target_ids.txt", "r") as f:
    target_ids = [int(line.strip()) for line in f]
len(target_ids)

180

## Ensure the citing cases are not already part of the SCOTUS set in review

In [11]:
scotus = pd.read_json("../experiments_624/data/scotus_citing_cited_sampled.json")
len(scotus)

21925

In [12]:
scotus_citing = list(set(scotus["citing_cluster_id"].to_list()))
len(scotus_citing)

482

In [13]:
assert len(set(citing_ids) - set(scotus_citing)) == len(citing_ids)

# Use Django Shell to get the citing cases metadata

In [14]:
with open('data/citing_opinions.json', 'r') as f:
    results = json.load(f)

In [15]:
records = []
for case_id, case_data in results.items():
    record = {'citing_cluster_id': int(case_id)}
    record.update({k: v for k, v in case_data.items()})
    
    opinion_filenames = [op['opinion_filename'] for op in case_data.get('opinion_data', [])]
    record['opinion_filenames'] = opinion_filenames
    
    records.append(record)

citing_df = pd.DataFrame(records)
citing_df.head()

,citing_cluster_id,citing_url,citing_court_id,citing_court_name,opinion_data,cited_cluster_ids,opinion_filenames
0,73856,https://www.courtlistener.com/opinion/73856/mo...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 73856, 'opinion_api': None, 'o...","[515532, 583488, 583996, 607973, 627502, 66580...",[73856_010combined.txt]
1,73857,https://www.courtlistener.com/opinion/73857/mo...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 73857, 'opinion_api': None, 'o...","[515532, 583996, 627502, 665807, 672459, 71585...",[73857_010combined.txt]
2,426024,https://www.courtlistener.com/opinion/426024/c...,ca1,Court of Appeals for the First Circuit,"[{'opinion_id': 426024, 'opinion_api': None, '...","[85267, 85530, 99087, 110017, 300790, 302312, ...",[426024_010combined.txt]
3,450672,https://www.courtlistener.com/opinion/450672/i...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 450672, 'opinion_api': None, '...","[97095, 100602, 260688, 264999, 356001, 110651...",[450672_010combined.txt]
4,475775,https://www.courtlistener.com/opinion/475775/i...,ca6,Court of Appeals for the Sixth Circuit,"[{'opinion_id': 475775, 'opinion_api': None, '...","[100898, 103240, 103393, 1486562, 1500608, 150...",[475775_010combined.txt]


## Get all cited case ids to a list for extracting the metadata

In [16]:
#with open('data/cited_ids.txt', 'w') as f:
#    for each in list(set(citing_df["cited_cluster_ids"].explode().dropna().tolist())):
#        f.write(f"{each}\n")

# Use Django Shell to extract the metadata for the cited cases

In [18]:
with open('data/cited_opinions.json', 'r') as f:
    results = json.load(f)

In [19]:
cited_metadata = pd.DataFrame.from_dict(results, orient='index')
cited_metadata = cited_metadata.reset_index().rename(columns={'index': 'cited_cluster_id'})
cited_metadata["cited_cluster_id"] = cited_metadata["cited_cluster_id"].astype(int)
cited_metadata.head()

,cited_cluster_id,cited_url,cited_court_id,cited_court_name,cited_case_name_short,cited_case_name,cited_case_name_full,cited_citations
0,67,https://www.courtlistener.com/opinion/67/news-...,ca4,Court of Appeals for the Fourth Circuit,,News & Observer Publishing Co. v. Raleigh-Durh...,The NEWS AND OBSERVER PUBLISHING COMPANY; The ...,[597 F.3d 570]
1,420,https://www.courtlistener.com/opinion/420/john...,ca2,Court of Appeals for the Second Circuit,In Re Johns-Manville Corp.,Johns-Manville Corp. v. Chubb Indemnity Insurance,"In Re JOHNS-MANVILLE CORPORATION, Debtor. John...","[2010 WL 1007832, 52 Bankr. Ct. Dec. (CRR) 266..."
2,635,https://www.courtlistener.com/opinion/635/ojed...,ca7,Court of Appeals for the Seventh Circuit,Ojeda,Ojeda v. Goldberg,"Ernest J. OJEDA and Beverly v. Ojeda, Defendan...","[2010 WL 1068216, 52 Bankr. Ct. Dec. (CRR) 267..."
3,131145,https://www.courtlistener.com/opinion/131145/b...,scotus,Supreme Court of the United States,Barnhart,Barnhart v. Thomas,"Barnhart, Commissioner of Social Security v. T...","[2003 U.S. LEXIS 8348, 540 U.S. 20, 124 S. Ct...."
4,131156,https://www.courtlistener.com/opinion/131156/k...,scotus,Supreme Court of the United States,Kontrick,Kontrick v. Ryan,Kontrick v. Ryan,"[2004 U.S. LEXIS 663, 540 U.S. 443, 124 S. Ct...."


In [20]:
len(cited_metadata)

21257

# Create result_df by merging citing and cited metadatas

In [21]:
result_df = citing_df.explode("cited_cluster_ids").reset_index(drop=True)
len(result_df)

37942

In [22]:
result_df = result_df.rename(columns={"cited_cluster_ids": "cited_cluster_id"})
result_df.head()

,citing_cluster_id,citing_url,citing_court_id,citing_court_name,opinion_data,cited_cluster_id,opinion_filenames
0,73856,https://www.courtlistener.com/opinion/73856/mo...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 73856, 'opinion_api': None, 'o...",515532,[73856_010combined.txt]
1,73856,https://www.courtlistener.com/opinion/73856/mo...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 73856, 'opinion_api': None, 'o...",583488,[73856_010combined.txt]
2,73856,https://www.courtlistener.com/opinion/73856/mo...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 73856, 'opinion_api': None, 'o...",583996,[73856_010combined.txt]
3,73856,https://www.courtlistener.com/opinion/73856/mo...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 73856, 'opinion_api': None, 'o...",607973,[73856_010combined.txt]
4,73856,https://www.courtlistener.com/opinion/73856/mo...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 73856, 'opinion_api': None, 'o...",627502,[73856_010combined.txt]


In [23]:
result_df = result_df.merge(cited_metadata, how="left", on="cited_cluster_id")
len(result_df)

37942

In [24]:
result_df.columns

Index(['citing_cluster_id', 'citing_url', 'citing_court_id',
       'citing_court_name', 'opinion_data', 'cited_cluster_id',
       'opinion_filenames', 'cited_url', 'cited_court_id', 'cited_court_name',
       'cited_case_name_short', 'cited_case_name', 'cited_case_name_full',
       'cited_citations'],
      dtype='object')

In [25]:
result_df = result_df[['citing_cluster_id', 'citing_url', 'citing_court_id',
       'citing_court_name', 'opinion_data', 'opinion_filenames', 
       'cited_cluster_id', 'cited_url', 'cited_court_id', 'cited_court_name',
       'cited_case_name_short', 'cited_case_name', 'cited_case_name_full',
       'cited_citations']]

In [26]:
result_df.head()

,citing_cluster_id,citing_url,citing_court_id,citing_court_name,opinion_data,opinion_filenames,cited_cluster_id,cited_url,cited_court_id,cited_court_name,cited_case_name_short,cited_case_name,cited_case_name_full,cited_citations
0,73856,https://www.courtlistener.com/opinion/73856/mo...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 73856, 'opinion_api': None, 'o...",[73856_010combined.txt],515532,https://www.courtlistener.com/opinion/515532/i...,ca11,Court of Appeals for the Eleventh Circuit,,In the Matter of Paul D. Folendore and Helen H...,In the Matter of Paul D. FOLENDORE and Helen H...,"[1989 WL 28, 18 Bankr. Ct. Dec. (CRR) 1247, 19..."
1,73856,https://www.courtlistener.com/opinion/73856/mo...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 73856, 'opinion_api': None, 'o...",[73856_010combined.txt],583488,https://www.courtlistener.com/opinion/583488/i...,ca11,Court of Appeals for the Eleventh Circuit,,,"In the Matter of Saybrook Manufacturing Co., I...","[23 Bankr. Ct. Dec. (CRR) 355, 1992 U.S. App. ..."
2,73856,https://www.courtlistener.com/opinion/73856/mo...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 73856, 'opinion_api': None, 'o...",[73856_010combined.txt],583996,https://www.courtlistener.com/opinion/583996/i...,ca7,Court of Appeals for the Seventh Circuit,,"In Re Henry E. Montoya, M.D. And Juanita F. Mo...","In Re Henry E. MONTOYA, M.D. and Juanita F. Mo...","[1992 WL 141941, 1992 U.S. App. LEXIS 14502, 7..."
3,73856,https://www.courtlistener.com/opinion/73856/mo...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 73856, 'opinion_api': None, 'o...",[73856_010combined.txt],607973,https://www.courtlistener.com/opinion/607973/i...,ca10,Court of Appeals for the Tenth Circuit,,"In Re William E. Richards, Debtor, United Stat...","In Re William E. RICHARDS, Debtor, UNITED STAT...","[1993 WL 182728, 1993 U.S. App. LEXIS 12648, 7..."
4,73856,https://www.courtlistener.com/opinion/73856/mo...,ca11,Court of Appeals for the Eleventh Circuit,"[{'opinion_id': 73856, 'opinion_api': None, 'o...",[73856_010combined.txt],627502,https://www.courtlistener.com/opinion/627502/i...,ca11,Court of Appeals for the Eleventh Circuit,,"In Re Empire for Him, Inc., Debtor. Capital Fa...","In Re EMPIRE FOR HIM, INC., Debtor. CAPITAL FA...","[1993 WL 325352, 24 Bankr. Ct. Dec. (CRR) 1103..."


## Tag the target cases from the target courts

In [27]:
result_df.loc[result_df["cited_cluster_id"].isin(target_ids), "cited_target"] = 1
result_df.loc[~result_df["cited_cluster_id"].isin(target_ids), "cited_target"] = 0

## Do some EDA

In [28]:
eda_cols = ['citing_cluster_id', 'citing_court_name', 'cited_cluster_id', 'cited_court_name', 'cited_target']

for col in eda_cols:
    print("----------")
    print(result_df[col].nunique())
    display(result_df[col].value_counts())

----------
1493


citing_cluster_id
8337992    292
1549737    261
1962505    260
2196109    247
1933158    199
          ... 
1872340      1
8518697      1
1865723      1
1535105      1
1936149      1
Name: count, Length: 1493, dtype: int64

----------
207


citing_court_name
United States Bankruptcy Court, E.D. Pennsylvania                 2501
United States Bankruptcy Court, N.D. Illinois                     1510
United States Bankruptcy Appellate Panel for the Ninth Circuit    1300
United States Bankruptcy Court, S.D. New York                     1297
United States Bankruptcy Court, N.D. Indiana                      1198
                                                                  ... 
District Court, W.D. Washington                                      8
Vermont Superior Court                                               8
District Court, W.D. North Carolina                                  7
District Court, S.D. Indiana                                         6
United States Bankruptcy Court, S.D. West Virginia                   2
Name: count, Length: 207, dtype: int64

----------
21257


cited_cluster_id
112522     146
1887361    122
1538334    101
110017      77
111722      76
          ... 
1810829      1
1826882      1
1829805      1
1847539      1
1457542      1
Name: count, Length: 21257, dtype: int64

----------
335


cited_court_name
Supreme Court of the United States                   4193
Court of Appeals for the Ninth Circuit               1510
Court of Appeals for the Seventh Circuit             1269
United States Bankruptcy Court, E.D. Pennsylvania    1265
Court of Appeals for the Fifth Circuit               1224
                                                     ... 
New York County Courts                                  1
Connecticut Superior Court                              1
Court of Common Pleas of Ohio, Hamilton County          1
Jackson County Court of Common Pleas                    1
City of Cleveland Municipal Court                       1
Name: count, Length: 335, dtype: int64

----------
2


cited_target
0.0    36427
1.0     1515
Name: count, dtype: int64

In [29]:
df_target = result_df[result_df["cited_target"] == 1]
print(df_target["cited_court_name"].nunique())
df_target["cited_court_name"].value_counts()

61


cited_court_name
United States Bankruptcy Court, E.D. Pennsylvania                  145
United States Bankruptcy Court, D. Utah                            122
United States Bankruptcy Court, N.D. Indiana                       121
United States Bankruptcy Court, D. Massachusetts                    80
United States Bankruptcy Appellate Panel for the Ninth Circuit      65
                                                                  ... 
United States Bankruptcy Court, N.D. Mississippi                     1
United States Bankruptcy Court, S.D. Mississippi                     1
United States Bankruptcy Appellate Panel for the Eighth Circuit      1
United States Bankruptcy Court, M.D. Pennsylvania                    1
United States Bankruptcy Court, E.D. California                      1
Name: count, Length: 61, dtype: int64

# Save the data for future use

In [30]:
result_df.to_json("data/bankruptcy_citing_cited.json")